# Fuentes de datos

Este notebook documenta todas las fuentes de datos utilizadas en la tesis, incluyendo información sobre la institución responsable, el propósito de los datos, su uso en el análisis, y las citas correspondientes. También describe las variables contenidas en cada dataset y los procesos de limpieza aplicados.

**Inputs necesarios:**
- Múltiples archivos de datos para referencia y verificación:
  - `fotocivicas-ubicacion-puntos/fotocivicas-ubicacion-puntos.shp`: Ubicación de cámaras
  - `vialidades.json`: Vialidades de la CDMX
  - `incidentes-viales.parquet`: Incidentes viales reportados
  - `afluencia-metro-cdmx.csv`: Afluencia del metro
  - `metro-station-coordinates.parquet`: Coordenadas de estaciones
  - `volumen-tránsito.xlsx`: Volumen de tránsito mensual

**Outputs generados:**
- Documentación de fuentes de datos (sin archivos guardados, solo documentación)

In [2]:
import os
import pandas as pd
import geopandas as gpd

PATH_DATA = '../../data'

### Radares de velociadad
- Fuente: `https://datos.cdmx.gob.mx/dataset/fotocivicas`
- Institución responsable: Secretaría de Seguridad Ciudadana (SSC)
- Propósito: Hacer de conocimiento público la ubicación de los radares.
- Última actualización: `6 de julio de 2023`
- Uso de estos datos en la tesis: Tratamiento
- Cita: Secretaría de Seguridad Ciudadana. (2023). *Fotocívicas (Ubicación)* [Data set]. Consultado Noviembre de 2025 en https://datos.cdmx.gob.mx/dataset/fotocivicas

#### Descripción de los datos
- Variables:
    - Vialidad sobre la que se encuentra el radar
    - El sentido sobre el que está la cámara dentro de esa vialidad
    - Latitud y longitud de la ubicación de las cámaras de velocidad
- Número de datos: **113**
- Limpieza: Ninguna

In [3]:
speed_cameras = gpd.read_file(
    os.path.join(
        PATH_DATA,
        'fotocivicas-ubicacion-puntos',
        'fotocivicas-ubicacion-puntos.shp'
    )
)

### Vialidades de la Ciudad de México
- Fuente: `https://datos.cdmx.gob.mx/dataset/vialidades-de-la-ciudad-de-mexico`
- Institución responsable: Secretaría de Movilidad (SEMOVI)
- Propósito: Informativo
- Última actualización: `4 de diciembre de 2023`
- Uso en la tesis: 
    - Genero cuadrícula y, para cada celda (mi unidad de tratamiento), contabilizo cruces viales en cada una
    - Restrinjo el análisis a las celdas que efectivamente se encuentran sobre vialidades principales
    - construyo las covariables que usaré como controles en el cálculo del propensity score y el matching
- Cita: Secretaría de Movilidad. (2023). *Vialidades primarias de la Ciudad de México* [Data set]. Consultado Noviembre de 2025 en https://datos.cdmx.gob.mx/dataset/vialidades-de-la-ciudad-de-mexico 

#### Descripción de los datos
- Variables: 
    - Nombre de la vialidad
    - Tipo de vía: vía primaria o de acceso controlado
    - Número de carriles
    - Nivel (zorder)
    - Circulación: si es doble sentido o de uno solo
    - Geometría: Traza en coordenadas de la vía
- Número de datos: **10,567**
- Limpieza: Ninguna

In [3]:
vialidades = gpd.read_file(os.path.join(PATH_DATA, 'vialidades.json'))

### Incidentes Viales
- Fuente: `https://datos.cdmx.gob.mx/dataset/incidentes-viales-c5`
- Institución:
    - Centro de Comando, Control, Cómputo, Comunicaciones y Contacto Ciudadano de la CDMX (C5)
    - El C5 de la CDMX es la entidad del Gobierno capitalino que recopila y analiza información para apoyar decisiones en seguridad pública, urgencias médicas, medio ambiente, protección civil, movilidad y servicios a la comunidad. Lo hace mediante videovigilancia, atención de llamadas y herramientas informáticas de inteligencia, con el objetivo de mejorar la calidad de vida de la ciudadanía.
- Propósito: 
    - El C5 de la CDMX construye la base de “incidentes viales” a partir de reportes que llegan sobre todo vía 911 (más de 80%), pero también por botones de auxilio, app, redes sociales, radios y videovigilancia. Cada reporte genera un folio con hora, lugar y tipo de hecho, y se clasifica como confirmado (afirmativo/informativo) o no confirmado (duplicado, falso o sin verificación); para análisis oficiales se usan solo los confirmados y, en prácticas de SEMOVI y PISVI, aquellos con “código_cierre” A o I. La confirmación puede ser en sitio por policías o por cámaras C2/C5, lo que mejora la confiabilidad. Los registros quedan georreferenciados como puntos y se publican para monitoreo y evaluación, aunque la base contabiliza eventos y no víctimas y no siempre distingue bien modalidades o severidad.
- Última actualización: `7 de marzo de 2024`
- Uso en la tesis: 
    - Es el outcome, sobre estos datos se hace la inferencia
- Cita: Centro de Comando, Control, Cómputo, Comunicaciones y Contacto Ciudadano de la CDMX. (2024). *Incidentes viales reportados por C5* [Data set]. Consultado noviembre de 2025 en https://datos.cdmx.gob.mx/dataset/incidentes-viales-c5

#### Descripción de los datos
- Variables: 
    - Fecha de creación
    - Hora de creación
    - Fecha de cierre
    - Hora de cierre
    - Tipo de Incidente C4
    - Incidente C4 -> especificación del tipo de incidente; e.g. tipo -- accidente, incidente -- choque sin lesionados
    - Código de cierre:
        - A - Afirmativo - Se especifica que estos son los datos confirmados por las autoridades en el lugar de los hechos
        - N - Negativo
        - I - Informativo
        - F - Falso
        - D - Duplicados
    - Clas con f alarma -> clasificación del incidente
    - Tipo entrada -> Medio por el cual se dio aviso del incidente
    - Coordenadas del incidente
- Número de datos: **2,115,080**
- Rango de fechas: **31 de diciembre de 2013 al 29 de febrero de 2024**

#### Limpieza
- Filtrado de datos:
    - Solo se mantienen registros con código de cierre A 
    - Se omiten aquellos registros en los que la clasificación del incidente sea `FALSA ALARMA` o `INCIDENTES EXTERNOS`
    - Se conservan únicamente datos en los que la variable `tipo_incidente_c4` sea *Accidente*, *Cadáver* o *Lesionado*
    - Solo se incluyen accidentes que hayan ocurrido entre 2016-04-22 y 2022-04-21, para tener una ventana de 3 años antes y 3 después de la implementación de las cámaras de velocidad
- Se categorizan los accidentes en una de las siguientes clases según su nivel de daño: 
    - FCS: fatales
        - Todos aquellos registros en los que la variable `tipo_incidente_c4` sea *Cadáver*
    - PIC: con lesiones personales
        - Todos aquellos registros que no sean fatales y la variable `clas_con_f_alarma` sea *Urgencia Médica*
    - MIN: menor, sin lesionados ni fallecidos
        - El resto de los accidentes

In [5]:
accidentes = (
    pd
    .read_parquet(os.path.join(PATH_DATA, 'incidentes-viales.parquet'))
    .reset_index(drop=True)
    .pipe(
        lambda df: 
        gpd
        .GeoDataFrame(
            data=df.drop(['latitud', 'longitud'], axis=1),
            geometry=gpd.points_from_xy(df.longitud, df.latitud)
        )
    )
)

### Afluencia en el metro
- Fuente: `https://datos.cdmx.gob.mx/dataset/afluencia-diaria-del-metro-cdmx`
- Institución:
    - Secretaría de Movilidad (SEMOVI)
- Propósito: Informativo
- Última actualización: `21 de octubre de 2025`
- Uso en la tesis:
    - Ponderar el número de incidentes con base en el flujo de gente en el metro como control por el posible cambio en movilidad durante la pandemia
- Cita: Secretaría de Movilidad. (2025). *Afluencia diaria del metro CDMX* [Data set]. Consultado noviembre de 2025 en https://datos.cdmx.gob.mx/dataset/afluencia-diaria-del-metro-cdmx

#### Descripción de los datos
- Variables
    - Fecha
    - Línea del metro
    - Estación
    - Afluencia : int
- Número de datos: **1,091,805**
- Rango de fechas: **1 de enero de 2010 al 30 de abril de 2025**
- Limpieza: Ninguna

In [ ]:
afluencia = pd.read_csv(os.path.join(PATH_DATA, "raw-data", "afluencia-metro-cdmx.csv"), encoding='utf-8')

### Coordenadas de las estaciones del metro de la CDMX
- Fuente: Google Maps
- Se creó buscando cada estación en Google Maps y registrando las coordenadas geográficas
- Uso en la tesis: 
    - Poder georeferenciar la ubicación de las estaciones del metro

#### Descripción de los datos
- Variables:
    - Estación: nombre de la estación
    - Coordenadas de la estación
- Número de datos: **167**

In [21]:
coordinates = pd.read_parquet(os.path.join(PATH_DATA, "metro-station-coordinates.parquet"))

### Volumen de tránsito mensual 
- Fuente: `https://micrs.sct.gob.mx/index.php/infraestructura/direccion-general-de-servicios-tecnicos/datos-viales`
- Institución: 
    - Secretaría de Infraestructura, Comunicaciones y Transportes
    - Instrumentar políticas públicas que impulsen el desarrollo económico, nacional y regional, mediante la expansión de infraestructura eficiente, sustentable, segura, incluyente y resiliente, así como la ampliación de la cobertura y mejora constante de la calidad, operación y oportunidad de los servicios de comunicaciones y transportes en beneficio de la población.
- Objetivo:
    - Con el objeto de conocer el comportamiento de las corrientes de tránsito durante todo el año, se instaló un conjunto de aparatos automáticos contadores de vehículos, distribuidos en diferentes tramos de la red carretera. Con este mismo propósito también se dispone de los volúmenes de tránsito que se registran en las plazas de cobro de Autopistas y Puentes de Cuota, que constituyen una de las fuentes más completas de información, en virtud de que su sistema de operación exige una clasificación detallada del tipo de vehículos que utilizan las obras a su cargo
- Uso en la tesis:
    - Ponderar el número de accidentes por el volumen de tránsito mensual para controlar por cambios en patrones de movilidad derivado de la pandemia por Covid 19
- Construcción de los datos:
    - En cada uno de los reportes de anuales se detalla el volumen de tránsito total en cada uno de los radares de conteo permanentes, así como el estado al que pertencen. Se tómo la información correspondiente a aquellos radares permanentes en la ciudad de méxico presentes durante todos los años estudiados.
    - Cabe destacar que hay un indicador para cada uno de los radares, sin embargo, este no es persistente a lo largo de los años, asumo que es porque van incorporando nuevos radares, por lo que se identificaron aquellos radares persistentes con base en la carretera, movimiento, caseta y coordenadas.
- Citas:
    - Secretaría de Comunicaciones y Transportes. (2015). *Datos Viales 2015*. Ciudad de México: Ruiz, G. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2015/Introduccion_DV_2015.pdf
    - Secretaría de Comunicaciones y Transportes. (2016). *Datos Viales 2016*. Ciudad de México: Ruiz, G. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2016/Introduccion_DV_2016.pdf
    - Secretaría de Comunicaciones y Transportes. (2017). *Datos Viales 2017*. Ciudad de México: Ruiz, G. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2017/Introduccion_DV_2017.pdf
    - Secretaría de Comunicaciones y Transportes. (2018). *Datos Viales 2018*. Ciudad de México: Ruiz, G. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2018/Introduccion_DV_2018.pdf
    - Secretaría de Comunicaciones y Transportes. (2019). *Datos Viales 2019*. Ciudad de México: Ruiz, G. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2019/00_INTRODUCCIÓN.pdf    
    - Secretaría de Comunicaciones y Transportes. (2020). *Datos Viales 2020*. Ciudad de México: Jiménez, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos-Viales-2020/00_Introducción_DV2020.pdf
    - Secretaría de Comunicaciones y Transportes. (2021). *Datos Viales 2021*. Ciudad de México: Jiménez, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos_Viales_2021/00_Introducción_DV2021.pdf
    - Secretaría de Infraestructura, Comunicaciones y Transportes. (2022). *Datos Viales 2022*. Ciudad de México: Arganis, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos_Viales_2022/00_Introducción_DV2022.pdf
    - Secretaría de Infraestructura, Comunicaciones y Transportes. (2023). *Datos Viales 2023*. Ciudad de México: Nuño, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos_Viales_2023/00_Introducción_DV2023.pdf
    - Secretaría de Infraestructura, Comunicaciones y Transportes. (2024). *Datos Viales 2024*. Ciudad de México: Nuño, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos_Viales_2024/00_Int_DV2024.pdf
    - Secretaría de Infraestructura, Comunicaciones y Transportes. (2025). *Datos Viales 2025*. Ciudad de México: Esteva, J. Accedido el 11 noviembre de 2025 en https://micrs.sct.gob.mx/images/DireccionesGrales/DGST/Datos_Viales_2025/00_DV2025_Introduccion.pdf
    - Dirección General de Servicios Técnicos. (2025). *Datos Viales*. Secretaría de Infraestructura, Comunicaciones y Transportes. https://micrs.sct.gob.mx/index.php/infraestructura/direccion-general-de-servicios-tecnicos/datos-viales
    
#### Descripción de los datos
- Volumen de tránsito
    - Variables
        - indicador
        - volumen total anual
        - latitud
        - longitud
        - porcentaje correspondiente al volumen de cada mes
        - año
    - Número de datos: **225**
    - Años: **2015 al 2024**
- Radares
    - Variables
        - indicador
        - carretera
        - movimiento
        - caseta
        - kilómetro
        - año
    - Número de datos: **225**
    - Años: **2015 al 2024**

In [18]:
volumen = pd.read_excel(
    os.path.join(PATH_DATA, "raw-data", "volumen-tránsito.xlsx"),
    sheet_name="volumen"
)

indicadores = pd.read_excel(
    os.path.join(PATH_DATA, "raw-data", "volumen-tránsito.xlsx"),
    sheet_name="indicadores"
)